In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install pyvi

In [ ]:
!nvidia-smi

In [ ]:
!python /kaggle/input/models/thnngvn/code-mtl/pytorch/default/1/train_mtl_acsa.py \
      --train_path /kaggle/input/datasets/thnngvn/res-vi/Train.txt --dev_path /kaggle/input/datasets/thnngvn/res-vi/Dev.txt \
      --test_path /kaggle/input/datasets/thnngvn/res-vi/Test.txt \
      --model_name vinai/phobert-base-v2 \
      --loss_weighting gradnorm \
      --num_attention_heads 8 \
      --adapter_dim 256 \
      --dropout 0.10 \
      --epochs 20 \
      --batch_size 16 \
      --eval_batch_size 32 \
      --encoder_lr 5e-5 \
      --head_lr 1e-4 \
      --weight_decay 0.01 \
      --warmup_ratio 0.06 \
      --patience 5 \
      --lambda_acd 0.7 \
      --lambda_sent 1.3 \
      --lambda_joint 1.0 \
      --seed 42 \
      --output_dir outputs/phobert_mtl_acsa

In [ ]:
import json
from pathlib import Path


def convert_prediction_json_to_txt(
    input_path: str,
    output_path: str,
):
    input_path = Path(input_path)
    output_path = Path(output_path)

    # =========================================================
    # Load JSON / JSONL
    # =========================================================
    if input_path.suffix.lower() == ".jsonl":
        samples = []

        with input_path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()

                if not line:
                    continue

                samples.append(json.loads(line))

    elif input_path.suffix.lower() == ".json":
        with input_path.open("r", encoding="utf-8") as f:
            samples = json.load(f)

        if isinstance(samples, dict):
            if "predictions" in samples:
                samples = samples["predictions"]
            elif "data" in samples:
                samples = samples["data"]
            else:
                raise ValueError(
                    "JSON phải là list hoặc chứa key 'predictions' / 'data'."
                )

    else:
        raise ValueError(
            f"Unsupported format: {input_path.suffix}"
        )

    output_path.parent.mkdir(parents=True, exist_ok=True)

    # =========================================================
    # Convert
    # =========================================================
    with output_path.open("w", encoding="utf-8") as f:

        for idx, sample in enumerate(samples, start=1):

            sample_id = sample.get(
                "id",
                sample.get("sample_id", idx),
            )

            text = sample.get(
                "text",
                sample.get(
                    "raw_text",
                    sample.get("sentence", ""),
                ),
            )

            predictions = sample.get(
                "prediction",
                sample.get(
                    "predictions",
                    sample.get("pred", []),
                ),
            )

            # Tránh trường hợp prediction = None
            if predictions is None:
                predictions = []

            labels = []

            # =================================================
            # Parse predictions
            # =================================================
            for pred in predictions:

                # tránh lỗi nếu phần tử không phải dict
                if not isinstance(pred, dict):
                    continue

                category = pred.get(
                    "category",
                    pred.get("aspect"),
                )

                sentiment = pred.get(
                    "sentiment",
                    pred.get("polarity"),
                )

                if category is None or sentiment is None:
                    continue

                sentiment = str(sentiment).lower().strip()

                # Không ghi NONE
                if sentiment in {
                    "none",
                    "absent",
                    "not_present",
                    "",
                }:
                    continue

                labels.append(
                    f"{{{category}, {sentiment}}}"
                )

            # =================================================
            # FALLBACK:
            # Nếu prediction rỗng / parse xong không còn label
            # =================================================
            if not labels:
                labels.append(
                    "{RESTAURANT#GENERAL, neutral}"
                )

            # =================================================
            # Write format giống Test.txt
            # =================================================
            f.write(f"#{sample_id}\n")
            f.write(str(text).strip() + "\n")
            f.write(", ".join(labels) + "\n\n")

    print(f"Converted: {len(samples)} samples")
    print(f"Output: {output_path}")

In [ ]:
convert_prediction_json_to_txt(
  input_path="outputs/phobert_mtl_acsa/test_predictions.jsonl",
  output_path="test_predictions.txt",
)

In [ ]:
import re
import sys

def get_labels_from_filename(filename):
    labels = []
    with open(filename, 'r', encoding = 'utf-8') as file:
        datasets = file.read()
        count = 0
        for line in datasets.split('\n'):
            if line != '':
                if count == 0:
                    count += 1
                elif count == 1:
                    count += 1
                elif count == 2:
                    labels.append(line.strip())
                    count = 0
        file.close()
    #print(len(labels))
    return labels

def clean_label(label):
    label = re.sub('[^A-Za-z#&]', '', label)
    label = re.sub('\\s+', ' ', label)
    return label

def convert_labels_to_dict(labels):
    dict_labels = []
    for label in labels:
        label_line = label.split('},')
        _dict = {}
        for objectLabel in label_line:
          try:
              aspect = clean_label(objectLabel.split(',')[0]).strip()
              polarity = clean_label(objectLabel.split(',')[1]).strip()
          except:
              print(label_line)
          _dict[aspect] = polarity
        dict_labels.append(_dict)
    return dict_labels

def get_common_attributeEntities(dict_labels):
    AttributeEntities = []
    for _dict in dict_labels:
        for key in _dict:
            if key not in AttributeEntities:
                AttributeEntities.append(key)
    AttributeEntities = sorted(AttributeEntities)
    return AttributeEntities

def get_aspects(dict_labels):
    aspects = []
    for _dict in dict_labels:
        for key in _dict:
            aspects.append(key)
    return aspects

def count_aspects(labels, Common_AttributeEntities):
    aspects = get_aspects(labels)
    num_aspects = [0] * len(Common_AttributeEntities)
    for aspect in aspects:
        num_aspects[Common_AttributeEntities.index(aspect)] += 1
    return num_aspects

def evaluation_labels(gold_labels, answer_labels, Common_AttributeEntities):
    num_aspect_gold = count_aspects(gold_labels, Common_AttributeEntities)
    num_aspect_answer = count_aspects(answer_labels, Common_AttributeEntities)
    correct_answer_aspects = [0] * len(Common_AttributeEntities)
    correct_answer_labels = [0] * len(Common_AttributeEntities)

    for i, _dict in enumerate(answer_labels):
        for key in _dict:
            if key in gold_labels[i].keys():
                correct_answer_aspects[Common_AttributeEntities.index(key)] += 1
                if answer_labels[i][key].strip() == gold_labels[i][key].strip():
                    correct_answer_labels[Common_AttributeEntities.index(key)] += 1
    #print('Correct Answer Aspects: ', correct_answer_aspects)
    #print('---------------------------------------------------')
    #print('Correct Answer Labels: ', correct_answer_labels)
    #print('---------------------------------------------------')
    #infor_evaluation(correct_answer_aspects, num_aspect_answer, num_aspect_gold, Common_AttributeEntities)
    #print('---------------------------------------------------')
    infor_evaluation(correct_answer_labels, num_aspect_answer, num_aspect_gold, Common_AttributeEntities)

def infor_evaluation(correct_answer, num_aspect_answer, num_aspect_gold, Common_AttributeEntities):
    for aspect in Common_AttributeEntities:
        if correct_answer[Common_AttributeEntities.index(aspect)] == 0:
            p = r = f = 0.0
        else:
            p = correct_answer[Common_AttributeEntities.index(aspect)] * 100 / num_aspect_answer[Common_AttributeEntities.index(aspect)]
            r = correct_answer[Common_AttributeEntities.index(aspect)] * 100 / num_aspect_gold[Common_AttributeEntities.index(aspect)]
            f = 2 * p * r / (p + r)
        print(aspect)
        print('%0.2f\t%0.2f\t%0.2f' % (p, r, f))
    p = sum(correct_answer) * 100 / sum(num_aspect_answer)
    r = sum(correct_answer) * 100 / sum(num_aspect_gold)
    f = 2 * p * r / (p + r)
    print('-------------------------------------------------------------')
    print('-------------------------------------------------------------')
    print('Mean Precision score: ', round(p,2))
    print('Mean Recall score: ', round(r,2))
    print('Mean F1 score: ', round(f,2))
    print('-------------------------------------------------------------')
    print('-------------------------------------------------------------')

def evaluation_system(gold_labels, answer_labels):
    gold_dicts = convert_labels_to_dict(gold_labels)
    answer_dicts = convert_labels_to_dict(answer_labels)
    AttributeEntities = get_common_attributeEntities(gold_dicts)
    #print('---------------INFORMATION FILE--------------------')
    #print('Aspect Name: ', AttributeEntities)
    #print("Aspect Gold: ", count_aspects(gold_dicts, AttributeEntities))
    #print("Aspect Answer: ", count_aspects(answer_dicts, AttributeEntities))
    #print('---------------------------------------------------')
    evaluation_labels(gold_dicts, answer_dicts, AttributeEntities)

def evaluation_system_by_file(file_gold, file_predict):
    gold_labels = get_labels_from_filename(file_gold)
    answer_labels = get_labels_from_filename(file_predict)
    evaluation_system(gold_labels, answer_labels)

In [ ]:
evaluation_system_by_file("/kaggle/input/datasets/thnngvn/res-vi/Test.txt", "test_predictions.txt")

## Colab: Hotel domain

Uses `--domain hotel`, which now trains from the original `Hotel_ABSA/{Train,Dev,Test}.txt` files
(raw text, same as how `restaurant` works) -- not ABSA_LLMs' `data/Pair/Hotel`. Its category codes
already match `mapper.py`'s `hotel_dict` keys exactly, so no translation is needed.
`--segmenter` defaults to `pyvi` for this domain since the text is raw/untokenized.

(Two annotation-line typos were fixed directly in `Hotel_ABSA/Train.txt` to make it parseable:
`{ROOMS#CLEANLINESS, negative, ROOMS#MISCELLANEOUS, positive}` -> two separate `{cat, sentiment}`
pairs, and a stray trailing comma in one `{SERVICE#GENERAL, positive, }` entry.)

If you want the ABSA_LLMs cleaned-text version instead (note: NOT the same train/dev/test split as
this one -- verified same underlying ~10k-example pool via label distribution, but re-shuffled),
use `--domain hotel_clean`, which is what this cell used to run.

Upload/mount the whole `ACSA` folder (containing `train_mtl_acsa.py`, `mapper.py`, `Hotel_ABSA/`,
and `data/Pair/`) to the Colab runtime first, e.g. via Google Drive mount or a zip upload, then `%cd`
into it.

In [ ]:
# Mount Drive (adjust the path to wherever the ACSA folder lives in your Drive),
# or instead upload ACSA.zip via the Colab file browser and !unzip it into /content.
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ACSA

!pip install -q pyvi transformers scikit-learn tqdm

In [ ]:
!python train_mtl_acsa.py \
      --domain hotel \
      --model_name vinai/phobert-base-v2 \
      --loss_weighting gradnorm \
      --num_attention_heads 8 \
      --adapter_dim 256 \
      --dropout 0.10 \
      --epochs 20 \
      --batch_size 16 \
      --eval_batch_size 32 \
      --encoder_lr 5e-5 \
      --head_lr 1e-4 \
      --weight_decay 0.01 \
      --warmup_ratio 0.06 \
      --patience 5 \
      --lambda_acd 0.7 \
      --lambda_sent 1.3 \
      --lambda_joint 1.0 \
      --seed 42 \
      --output_dir outputs/phobert_mtl_acsa_hotel

### Per-category evaluation (works for any domain)

`test_predictions.jsonl` already stores `gold` and `prediction` together per sample (written by
`train_mtl_acsa.py`'s `write_predictions`), so this evaluates directly from it -- no external gold
`.txt` file or category-string regex parsing needed, unlike `evaluation_system_by_file` above which
only works for Restaurant's original raw file layout.

In [ ]:
import json
from collections import defaultdict


def evaluate_jsonl_per_category(jsonl_path):
    gold_counts = defaultdict(int)
    pred_counts = defaultdict(int)
    correct_counts = defaultdict(int)

    with open(jsonl_path, encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            gold = {g["category"]: g["sentiment"] for g in record["gold"]}
            pred = {p["category"]: p["sentiment"] for p in record["prediction"]}

            for cat in gold:
                gold_counts[cat] += 1
            for cat, sent in pred.items():
                pred_counts[cat] += 1
                if gold.get(cat) == sent:
                    correct_counts[cat] += 1

    categories = sorted(set(gold_counts) | set(pred_counts))
    for cat in categories:
        c, p, g = correct_counts[cat], pred_counts[cat], gold_counts[cat]
        prec = 100 * c / p if p else 0.0
        rec = 100 * c / g if g else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
        print(cat)
        print("%0.2f\t%0.2f\t%0.2f" % (prec, rec, f1))

    tot_c, tot_p, tot_g = sum(correct_counts.values()), sum(pred_counts.values()), sum(gold_counts.values())
    P = 100 * tot_c / tot_p if tot_p else 0.0
    R = 100 * tot_c / tot_g if tot_g else 0.0
    F = 2 * P * R / (P + R) if (P + R) else 0.0
    print("-" * 60)
    print("Mean Precision score: ", round(P, 2))
    print("Mean Recall score: ", round(R, 2))
    print("Mean F1 score: ", round(F, 2))
    return P, R, F

In [ ]:
evaluate_jsonl_per_category("outputs/phobert_mtl_acsa_hotel/test_predictions.jsonl")

## Colab: Restaurant, cleaned-text A/B

Same 7028/771/1938 samples and labels as the original Restaurant run above, but text comes from
`data/Pair/Restaurant` (`clean_doc()`-preprocessed: lowercased, slang-normalized, numbers masked to
`num`) instead of the raw `Res_ABSA/*.txt` files. Everything else (hyperparameters, seed) is kept
identical to the baseline cell so `outputs/phobert_mtl_acsa/metrics.json` (raw, 77.43 test F1) and
`outputs/phobert_mtl_acsa_restaurant_clean/metrics.json` (cleaned) are directly comparable.

In [ ]:
!python train_mtl_acsa.py \
      --domain restaurant_clean \
      --model_name vinai/phobert-base-v2 \
      --loss_weighting gradnorm \
      --num_attention_heads 8 \
      --adapter_dim 256 \
      --dropout 0.10 \
      --epochs 20 \
      --batch_size 16 \
      --eval_batch_size 32 \
      --encoder_lr 5e-5 \
      --head_lr 1e-4 \
      --weight_decay 0.01 \
      --warmup_ratio 0.06 \
      --patience 5 \
      --lambda_acd 0.7 \
      --lambda_sent 1.3 \
      --lambda_joint 1.0 \
      --seed 42 \
      --output_dir outputs/phobert_mtl_acsa_restaurant_clean

In [ ]:
evaluate_jsonl_per_category("outputs/phobert_mtl_acsa_restaurant_clean/test_predictions.jsonl")

## Colab: Hotel, enhanced training config

`--domain hotel` (raw `Hotel_ABSA` text, with the two typo fixes) already beats `hotel_clean`:
**75.13 test F1** (P=75.39, R=74.88) vs `hotel_clean`'s 74.66 -- so this builds on `hotel`, not
`hotel_clean`. The per-category breakdown from that `hotel` run makes the case for these changes
directly: 3 categories dead at 0.00 F1 (`FACILITIES#COMFORT`, `ROOMS#MISCELLANEOUS`,
`ROOM_AMENITIES#PRICES`), several more (`HOTEL#MISCELLANEOUS`=17.65, `FACILITIES#MISCELLANEOUS`=26.67,
`ROOM_AMENITIES#COMFORT`=30.00) far below the 75-93 range most categories sit in -- classic
low-support tail categories a single global threshold and plain CE/BCE underserve.

Same hyperparameters as the `hotel` baseline cell, plus:
- `--gate_lr 5e-3` + separates the ACD->sentiment gate into its own optimizer group with zero
  weight_decay (logged `gate=0.50` unmoving for entire runs -- weight_decay was actively pulling it
  back toward inactive gating while sharing head_lr/weight_decay with everything else)
- `--llrd_decay 0.9`: layer-wise LR decay on the PhoBERT encoder (top layer gets full `encoder_lr`,
  lower layers/embeddings get progressively less)
- `--acd_loss_fn focal --sent_loss_fn focal --joint_loss_fn focal --focal_gamma 2.0`: focal loss on
  all three heads instead of plain BCE/CE, targeting exactly the dead/weak tail categories above
- `--per_category_threshold`: tunes a separate ACD presence threshold per category on dev instead
  of one global threshold -- free macro-F1 gain, pure post-hoc reranking

Check `train.log` in the output dir for the per-epoch `gate_alpha=...` line (now logged at 6 decimal
places) to see whether it's actually moving now.

In [ ]:
!python train_mtl_acsa.py \
      --domain hotel \
      --model_name vinai/phobert-base-v2 \
      --loss_weighting gradnorm \
      --num_attention_heads 8 \
      --adapter_dim 256 \
      --dropout 0.10 \
      --epochs 20 \
      --batch_size 16 \
      --eval_batch_size 32 \
      --encoder_lr 5e-5 \
      --head_lr 1e-4 \
      --gate_lr 5e-3 \
      --llrd_decay 0.9 \
      --weight_decay 0.01 \
      --warmup_ratio 0.06 \
      --patience 5 \
      --lambda_acd 0.7 \
      --lambda_sent 1.3 \
      --lambda_joint 1.0 \
      --acd_loss_fn focal \
      --sent_loss_fn focal \
      --joint_loss_fn focal \
      --focal_gamma 2.0 \
      --per_category_threshold \
      --seed 42 \
      --output_dir outputs/phobert_mtl_acsa_hotel_v2

In [ ]:
evaluate_jsonl_per_category("outputs/phobert_mtl_acsa_hotel_v2/test_predictions.jsonl")